# Lab 3 — A chat UI that can actually do something

**~50 minutes.** Ed's Week 2 project — a multi-modal customer-support assistant with a
Gradio UI and function calling — rebuilt free-tier.

The point of this lab is the **tool-calling handshake**. Everything called an "agent" is
this loop plus ambition:

    model -> "call get_price(destination='Berlin')" -> your code runs it -> result back -> model answers

Your code executes the function. The model only asks. That boundary is where permissions,
validation and audit live.

In [ ]:
import json
import gradio as gr
from shared import client, model_name, MODEL

PRICES = {"berlin": 499, "lisbon": 620, "tokyo": 1400, "reykjavik": 780}
SEATS  = {"berlin": 3, "lisbon": 0, "tokyo": 12, "reykjavik": 5}


def get_ticket_price(destination: str) -> str:
    """Return the return fare in USD for a destination city."""
    price = PRICES.get(destination.strip().lower())
    return json.dumps({"destination": destination,
                       "price_usd": price,
                       "found": price is not None})


def check_availability(destination: str) -> str:
    """Return how many seats are left."""
    seats = SEATS.get(destination.strip().lower())
    if seats is None:
        return json.dumps({"destination": destination, "found": False})
    return json.dumps({"destination": destination, "seats_left": seats,
                       "sold_out": seats == 0, "found": True})


print(get_ticket_price("Berlin"), check_availability("Lisbon"))

## 1. Describe the tools to the model

A tool definition is a name, a description, and a JSON schema of parameters. **The
description is a prompt** — write it for a competent new hire: what it does, when to use it,
what it returns.

In [ ]:
price_schema = {
    "type": "function",
    "function": {
        "name": "get_ticket_price",
        "description": ("Return the current return fare in USD for a destination city. "
                        "Call this whenever the customer asks about price or cost. "
                        "Returns found=false for cities we do not serve."),
        "parameters": {
            "type": "object",
            "properties": {"destination": {"type": "string",
                                           "description": "Destination city, e.g. Berlin"}},
            "required": ["destination"],
        },
    },
}

availability_schema = {
    "type": "function",
    "function": {
        "name": "check_availability",
        "description": ("Return how many seats remain for a destination. Call before "
                        "encouraging a booking. Returns sold_out=true when zero seats remain."),
        "parameters": {
            "type": "object",
            "properties": {"destination": {"type": "string"}},
            "required": ["destination"],
        },
    },
}

TOOLS = [price_schema, availability_schema]
TOOL_IMPL = {"get_ticket_price": get_ticket_price, "check_availability": check_availability}

## 2. The handshake

One call, check for `tool_calls`, run them, append the results as `role: "tool"` messages,
call again. Note the loop — a model may call two tools before answering, and a good one
often does.

In [ ]:
SYSTEM = """You are a support assistant for a small airline. Keep answers to two or three
sentences. Never invent a price or a seat count — always call the tool. If we do not serve a
city, say so plainly and suggest one we do serve."""


def as_messages(history: list) -> list:
    """Gradio 5+ hands back OpenAI-style dicts; older versions hand back (user, assistant)
    pairs. Normalising here means the same function works on any version."""
    out = []
    for turn in history or []:
        if isinstance(turn, dict):
            out.append({"role": turn["role"], "content": turn["content"]})
        else:
            user, assistant = turn
            if user:
                out.append({"role": "user", "content": user})
            if assistant:
                out.append({"role": "assistant", "content": assistant})
    return out


def respond(message: str, history: list) -> str:
    messages = [{"role": "system", "content": SYSTEM}] + as_messages(history)
    messages.append({"role": "user", "content": message})

    for _ in range(5):                       # step cap: never loop forever
        reply = client().chat.completions.create(
            model=model_name(), messages=messages, tools=TOOLS, temperature=0.2,
        ).choices[0].message

        if not reply.tool_calls:
            return reply.content or "(no content)"

        messages.append(reply)
        for call in reply.tool_calls:
            fn = TOOL_IMPL.get(call.function.name)
            try:
                args = json.loads(call.function.arguments or "{}")
                result = fn(**args) if fn else json.dumps({"error": "unknown tool"})
            except Exception as exc:          # instructive errors, not stack traces
                result = json.dumps({"error": f"{type(exc).__name__}: {exc}"})
            print(f"  [tool] {call.function.name}({call.function.arguments}) -> {result}")
            messages.append({"role": "tool", "tool_call_id": call.id, "content": result})

    return "I could not complete that within the step limit — please rephrase."


print(respond("How much is a ticket to Berlin, and can I still get a seat?", []))

**Tool calling is where free models wobble.** If yours ignores the tools or emits a
malformed call, that is the lesson of this lab, not a broken notebook: tool-use quality is a
real axis of model selection. Fixes, in order — set `temperature=0`, sharpen the tool
description, switch `MODEL` in `.env` to another free model that lists tool support, or run
this one lab against a paid model if you have a key.

## 3. Put a UI on it

Gradio turns the function into a web app in one line. This is why Ed uses it: no front-end
work between you and a demo someone else can click.

In [ ]:
# Gradio 6 removed the `type=` argument (OpenAI-style messages are now the only format).
# If you are on Gradio 4, add type="messages" here.
demo = gr.ChatInterface(fn=respond, title="Airline assistant",
                        examples=["How much to Tokyo?", "Any seats to Lisbon?",
                                  "What about Cairo?"])
demo.launch(inbrowser=True)   # interrupt the kernel to stop the server

## Stretch goals

1. **A write tool with a gate.** Add `book_seat(destination, passenger)` that mutates
   `SEATS`. Then require confirmation: the model proposes, your code asks the human, and only
   an explicit yes executes. This is the human-in-the-loop pattern, and it is the control
   that actually holds in production.
2. **A bad tool description.** Rewrite one description to be vague ("gets info about a
   city"). Watch the model call it at the wrong times. Restore it. That contrast is worth
   more than any lecture on tool design.
3. **Streaming.** Convert `respond` to a generator and yield partial content so the UI types.
4. **Multimodal.** If your model supports images, accept an uploaded boarding-pass photo and
   have the model read the destination off it — Ed's version of this project adds image and
   audio in exactly this spot.
5. **Log the trace.** Print every message list before each call. When an agent misbehaves in
   Lab 5, this transcript is your stack trace.